Tagging and extraction Using Openai Tools

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
openai.api_key = os.getenv("OPENAI_API_KEY")

In [2]:
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from langchain_classic.utils.openai_functions import convert_pydantic_to_openai_function

In [3]:
class Tagging(BaseModel):
    sentiment: str = Field(description="The sentiment of the text")
    language: str = Field(description="The language of the text")

In [4]:
convert_pydantic_to_openai_function(Tagging)

{'name': 'Tagging',
 'description': '',
 'parameters': {'properties': {'sentiment': {'description': 'The sentiment of the text',
    'type': 'string'},
   'language': {'description': 'The language of the text', 'type': 'string'}},
  'required': ['sentiment', 'language'],
  'type': 'object'}}

In [5]:
class Tagging(BaseModel):
    """
    This is a tagging model that tags the text with sentiment and language.
    """
    sentiment: str = Field(description="The sentiment of the text")
    language: str = Field(description="The language of the text")

In [6]:
convert_pydantic_to_openai_function(Tagging)

{'name': 'Tagging',
 'description': 'This is a tagging model that tags the text with sentiment and language.',
 'parameters': {'properties': {'sentiment': {'description': 'The sentiment of the text',
    'type': 'string'},
   'language': {'description': 'The language of the text', 'type': 'string'}},
  'required': ['sentiment', 'language'],
  'type': 'object'}}

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [8]:
model = ChatOpenAI(
    model="LongCat-Flash-Chat",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai"  # 兼容 OpenAI 的第三方 API
)

In [9]:
tagged_functions = [convert_pydantic_to_openai_function(Tagging)]

In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that tags text with sentiment and language, and return json format."),
    ("user", "{text}"),
])


In [11]:
model_with_functions = model.bind(
    functions=tagged_functions,
    function_call={"name": "tagging"},
)


In [12]:
chain = prompt | model_with_functions

In [13]:
chain.invoke("I'm very happy today!")


AIMessage(content='{\n  "sentiment": "positive",\n  "language": "en"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 36, 'total_tokens': 54, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'longcat-flash-chatai-api', 'system_fingerprint': None, 'id': 'dbbac8e086f44fa89a7f94c452050de9', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1101-df2e-7ea2-96d9-4d8b8d8111ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 18, 'total_tokens': 54, 'input_token_details': {}, 'output_token_details': {}})

In [14]:
chain.invoke("I'm fucking sad today!")

AIMessage(content='{\n  "sentiment": "negative",\n  "language": "en"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 36, 'total_tokens': 54, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'longcat-flash-chatai-api', 'system_fingerprint': None, 'id': '1ec6a70ec78e4ba1aba965489e6b310b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1101-e154-76c3-8fbd-42b549bb265e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 18, 'total_tokens': 54, 'input_token_details': {}, 'output_token_details': {}})

In [15]:
from langchain_core.output_parsers import JsonOutputParser

In [16]:
chain = prompt | model_with_functions | JsonOutputParser()

In [17]:
chain.invoke("I'm very happy today!")

{'sentiment': 'positive', 'language': 'en'}

In [18]:
from typing import Optional
class Person(BaseModel):
    """
    This is a information about a person
    """
    name: str = Field(description="The name of the person")
    age: Optional[int] = Field(description="The age of the person")

In [19]:
class Info(BaseModel):
    "Informathon to extract"
    person: Person = Field(description="List of information about a person")

In [20]:
convert_pydantic_to_openai_function(Info)

{'name': 'Info',
 'description': 'Informathon to extract',
 'parameters': {'properties': {'person': {'description': 'List of information about a person',
    'properties': {'name': {'description': 'The name of the person',
      'type': 'string'},
     'age': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
      'description': 'The age of the person'}},
    'required': ['name', 'age'],
    'type': 'object'}},
  'required': ['person'],
  'type': 'object'}}

In [21]:
extract_functions = [convert_pydantic_to_openai_function(Info)]
extract_model = model.bind(
    functions=extract_functions,
    function_call={"name": "Info"},
)

In [22]:
extract_model.invoke("Joe is 20 years old, his mom is 45 years old")

AIMessage(content="Joe is 20 years old, and his mom is 45 years old. That means she was **25 years old** when Joe was born (45 − 20 = 25).\n\nThis is a typical age gap for a parent and child, and it’s perfectly normal! Let me know if you'd like to explore anything else—like how their ages will compare in the future, or fun facts about age differences. 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 27, 'total_tokens': 123, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'longcat-flash-chatai-api', 'system_fingerprint': None, 'id': '83a15d132ba8431eaf3ef2f5f0995305', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1101-ef5b-7573-b93d-9616246f7063-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 

In [23]:
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, return json format"),
    ("user", "{text}"),
])


In [24]:
chain = extract_prompt | extract_model | JsonOutputParser()


In [25]:
chain.invoke({"text": "Joe is 20 years old, his mom is 45 years old"})

{'Joe_age': 20, 'mom_age': 45}

In [26]:
chain.invoke("Joe is 20 years old")


{'name': 'Joe', 'age': 20}

In [27]:
from langchain_core.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

In [28]:
chain = extract_prompt | extract_model | JsonKeyOutputFunctionsParser(key_name = "people")

In [29]:
chain.invoke({"text": "Joe is 20 years old, his mom is 45 years old"})

OutputParserException: Could not parse function call: 'function_call'
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 